# Task 3: Multimodal Housing Price Prediction
CNN image features + tabular data fusion.

In [ ]:
"""
Task 3: Multimodal ML – Housing Price Prediction Using Images + Tabular Data
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing import image

# -------------------------
# Load tabular dataset
# -------------------------
df = pd.read_csv("housing_data.csv")

# Example columns:
# bedrooms, bathrooms, area, price, image_path

X_tabular = df[["bedrooms", "bathrooms", "area"]]
y = df["price"]

# -------------------------
# Image feature extractor
# -------------------------
cnn_model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")

def extract_features(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    features = cnn_model.predict(img_array, verbose=0)
    return features.flatten()

image_features = np.array([extract_features(path) for path in df["image_path"]])

# Combine image + tabular features
combined_features = np.hstack((X_tabular.values, image_features))

X_train, X_test, y_train, y_test = train_test_split(
    combined_features, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions) ** 0.5

print("MAE:", mae)
print("RMSE:", rmse)
